## Audio Exploration

#### Importamos dependencias

In [11]:
import json
import numpy as np
import matplotlib.pyplot as plt
import librosa                  # per MFCC
from wav2vec import cutvowel, wav2vec
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


#### Cargamos nuestro json y nuestro archivo de audio .wav

In [12]:
with open("./vowels/alex.json") as f:
    data = json.load(f)

print("Número de segments:", len(data))
print(data[30])

# --------------------------
# Extracció de característiques
# --------------------------
X = []
y = []

wav_file = "./vowels/alex.wav"

Número de segments: 138
{'vocal': 'A', 'start': '14.55', 'end': '14.65'}


Recorremos cada segmento de audio, lo recortamos y extraemos características:
datos con wav2vec + MFCCs.
Luego las combinamos en un vector y guardamos su etiqueta.

In [13]:
for i in range(len(data)):
    start = float(data[i]["start"])
    end   = float(data[i]["end"])
    vocal = data[i]["vocal"]

    Fs, cut = cutvowel(wav_file, start, end)
    
    if len(cut) < 100:
        continue

    # MFCCs + wav2vec combinats
    vec_wav2vec = wav2vec(cut, Fs)

    # MFCC amb librosa
    cut_float = cut.astype(float)
    mfccs = librosa.feature.mfcc(y=cut_float, sr=Fs, n_mfcc=13)
    mfcc_mean = mfccs.mean(axis=1)

    # combinem característiques
    vec = np.concatenate([vec_wav2vec, mfcc_mean])
    X.append(vec)
    y.append(vocal)

X = np.array(X)
y = np.array(y)


Mostramos el tamaño de los datos:
X tiene 138 muestras con 16 características cada una,
y y tiene 138 etiquetas.

In [14]:
print("Dimensions X:", X.shape)
print("Dimensions y:", y.shape)

Dimensions X: (138, 16)
Dimensions y: (138,)
